In [ ]:
# Cell 1: Core AI Pathfinding Logic
import heapq

class Node:
    def __init__(self, position, parent=None):
        self.position = position
        self.parent = parent
        self.g = 0 # Distance from start
        self.h = 0 # Estimated distance to target
        self.f = 0 # Total cost (g + h)

    def __lt__(self, other):
        return self.f < other.f

def astar_search(grid, start, end):
    open_list = []
    closed_set = set()
    
    start_node = Node(start)
    end_node = Node(end)
    heapq.heappush(open_list, start_node)

    # Possible movements: Up, Down, Left, Right
    directions = [(0, -1), (0, 1), (-1, 0), (1, 0)]

    while open_list:
        current_node = heapq.heappop(open_list)
        closed_set.add(current_node.position)

        # If we reached the package!
        if current_node.position == end_node.position:
            path = []
            while current_node:
                path.append(current_node.position)
                current_node = current_node.parent
            return path[::-1] # Return reversed path

        # Check all 4 surrounding squares
        for dir in directions:
            node_pos = (current_node.position[0] + dir[0], current_node.position[1] + dir[1])

            # Ensure within grid bounds
            if node_pos[0] > (len(grid) - 1) or node_pos[0] < 0 or node_pos[1] > (len(grid[len(grid)-1]) - 1) or node_pos[1] < 0:
                continue

            # Ensure walkable terrain (0 = clear, 1 = wall/obstacle)
            if grid[node_pos[0]][node_pos[1]] != 0:
                continue
                
            if node_pos in closed_set:
                continue

            # Create new node
            new_node = Node(node_pos, current_node)
            new_node.g = current_node.g + 1
            # Manhattan distance heuristic (perfect for grid movement)
            new_node.h = abs(node_pos[0] - end_node.position[0]) + abs(node_pos[1] - end_node.position[1])
            new_node.f = new_node.g + new_node.h

            # Check if this path is better than existing ones
            if add_to_open(open_list, new_node):
                heapq.heappush(open_list, new_node)

    return None # Return None if no path exists

def add_to_open(open_list, neighbor):
    for node in open_list:
        if neighbor.position == node.position and neighbor.g >= node.g:
            return False
    return True

In [ ]:
# Cell 2: Visualization Function
def display_warehouse(grid, path, start, end):
    symbols = {
        0: "⬜", # Empty aisle
        1: "⬛", # Shelf / Obstacle
        'S': "🤖", # Robot Start
        'E': "📦", # Package Target
        'P': "🟩"  # Path
    }
    
    # Create a copy of the grid for drawing
    display_grid = [row[:] for row in grid]
    
    # Draw the path
    if path:
        for step in path:
            display_grid[step[0]][step[1]] = 'P'
            
    # Draw Start and End
    display_grid[start[0]][start[1]] = 'S'
    display_grid[end[0]][end[1]] = 'E'

    print("\nFACTORY FLOOR MAP:")
    print("=" * 25)
    for row in display_grid:
        line = "".join([symbols.get(cell, symbols[cell]) if type(cell) == int else symbols[cell] for cell in row])
        print(line)
    print("=" * 25)
    
    if path:
        print(f"✅ Route Optimized! It will take {len(path)-1} steps.")
    else:
        print("❌ ALERT: Bot is completely blocked. Robot requires assistance.")

In [ ]:
# Cell 3: Live Test

warehouse_grid = [
    [0, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 1, 1, 1, 0, 0, 0],
    [1, 0, 0, 0, 1, 0, 1, 0],
    [0, 0, 1, 0, 1, 0, 1, 0],
    [0, 0, 1, 0, 0, 0, 1, 0],
    [0, 0, 1, 1, 1, 1, 1, 0],
    [0, 0, 0, 0, 0, 0, 0, 0]
]

start_position = (0, 0)   
target_package = (4, 3)   

print("Running AI Route Optimizer...")
fastest_path = astar_search(warehouse_grid, start_position, target_package)

display_warehouse(warehouse_grid, fastest_path, start_position, target_package)

Running AI Route Optimizer...

FACTORY FLOOR MAP:
🤖⬜⬜⬜⬜⬜⬜⬜
🟩⬛⬛⬛⬛⬜⬜⬜
🟩🟩🟩🟩⬛⬜⬛⬜
⬜⬜⬛🟩⬛⬜⬛⬜
⬜⬜⬛📦⬜⬜⬛⬜
⬜⬜⬛⬛⬛⬛⬛⬜
⬜⬜⬜⬜⬜⬜⬜⬜
✅ Route Optimized! It will take 7 steps.
